# Inférence Avancée : TTA et Calibration du Seuil

Ce notebook fusionne l'application de la **Test Time Augmentation (TTA)** et la **calibration du seuil de décision** pour optimiser les performances finales sur le projet PINKCC.

In [1]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset

# Configuration du root projet
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)

Project root: /home/alouiyaz/projects/PINKCC/Brain_tumor_MRI_classification


In [2]:
from pinkcc_ct_seg.data.dataset import BrainMRIDataset
from pinkcc_ct_seg.data.transforms import get_eval_transforms, get_tta_transforms
from pinkcc_ct_seg.data.split import make_train_val_split
from pinkcc_ct_seg.models.resnet18 import build_resnet18
from pinkcc_ct_seg.training.engine import predict_probabilities
from pinkcc_ct_seg.evaluation.tta import predict_probabilities_tta
from pinkcc_ct_seg.evaluation.thresholding import (
    scan_thresholds,
    select_threshold_by_best_f1,
    select_threshold_by_recall_constraint,
    apply_threshold
)
from pinkcc_ct_seg.utils import set_seed, get_device, load_checkpoint

set_seed(42)
device = get_device()

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
DATA_DIR = PROJECT_ROOT / "data/raw/brain_mri/Training"
TEST_DIR = PROJECT_ROOT / "data/raw/brain_mri/Testing"
MODEL_PATH = PROJECT_ROOT / "artifacts/checkpoints/best_resnet18_methodB_advanced.pt"

print("Device:", device)
print("Model Path exists:", MODEL_PATH.exists())

Device: cpu
Model Path exists: True


In [4]:
eval_tfms = get_eval_transforms(img_size=IMG_SIZE)
tta_tfms = get_tta_transforms(img_size=IMG_SIZE)

full_train_ds = BrainMRIDataset(DATA_DIR, transform=eval_tfms)
labels = [full_train_ds.samples[i][1] for i in range(len(full_train_ds))]

train_idx, val_idx = make_train_val_split(labels, val_size=0.2, random_state=42)
val_ds = Subset(full_train_ds, val_idx)
test_ds = BrainMRIDataset(TEST_DIR, transform=eval_tfms)

print(f"Validation size: {len(val_ds)} | Test size: {len(test_ds)}")

Validation size: 574 | Test size: 394


In [5]:
model = build_resnet18(num_classes=2, pretrained=False)
model = load_checkpoint(model, MODEL_PATH, map_location=device)
model = model.to(device).eval()
print("Modèle chargé.")

TypeError: load_checkpoint() got an unexpected keyword argument 'map_location'

In [ ]:
print("Application de la TTA sur le set de validation...")
y_val, probs_val_tta = predict_probabilities_tta(
    model, 
    val_ds, 
    device, 
    tta_tfms
)

In [ ]:
results = scan_thresholds(y_val, probs_val_tta)

best_f1_res = select_threshold_by_best_f1(results)
best_recall_res = select_threshold_by_recall_constraint(results, min_recall=0.90)

print(f"Seuil Optimal (F1): {best_f1_res['threshold']:.4f}")
print(f"Seuil Médical (Recall > 90%): {best_recall_res['threshold']:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot([r['threshold'] for r in results], [r['f1'] for r in results], label='F1-Score')
plt.plot([r['threshold'] for r in results], [r['recall'] for r in results], label='Recall')
plt.axvline(best_f1_res['threshold'], color='r', linestyle='--', label='Best F1')
plt.title("Calibration du seuil après TTA")
plt.xlabel("Seuil")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
final_threshold = best_f1_res['threshold']
print(f"--- ÉVALUATION FINALE SUR TEST (Seuil: {final_threshold:.4f}) ---")

y_test, probs_test_tta = predict_probabilities_tta(model, test_ds, device, tta_tfms)
y_pred_test = apply_threshold(probs_test_tta, final_threshold)

print(classification_report(y_test, y_pred_test, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_test))